# UnsafeNet video counterfactual feasibility test

**One source clip, one prompt-guided edit, on a free Colab GPU.**

Source: `0_tr143.mp4` from *Video Dataset for Safe and Unsafe Behaviours*
(doi:10.17632/xjmtb22pff.1, CC BY 4.0), the dataset behind
Önal & Dandıl, *Unsafe-Net*, Multimedia Tools and Applications (2025) 84:34967–34993.

Edit: class **0 Safe Walkway Violation** toward class **4 Safe Walkway**.
The worker should move from the bare concrete aisle onto the green marked walkway.

### What this notebook is not

- **Not LD-ViCE.** LD-ViCE fine-tunes CogVideoX per dataset and steers denoising with
  target-model gradients, SmoothGrad and an artifact refinement stage. This notebook uses a
  pretrained checkpoint with text conditioning only. It is a prompt-guided editing baseline.
- **Not a safety verdict.** Nothing here establishes that an output clip depicts safe
  behaviour. Outputs are **candidate counterfactuals** until a human reviews them against
  `templates/REVIEW_TEMPLATE.md`.
- **Not classifier-evaluated.** The Unsafe-Net authors released no code and no weights, so
  there is no target classifier to flip. Stage 5 documents that blocker; it does not run.

### Stages

| Cell group | Stage |
|---|---|
| 1–3 | Hardware, environment, pinned dependencies (recorded before any weights load) |
| 4–5 | Fetch the source clip, preprocess to 720×480×49 |
| 6 | Stage 3: VAE-only reconstruction check |
| 7–8 | Stage 4: prompt-guided source-video edit, comparison, manifest |
| 9 | Stage 5: classifier-guidance gate (blocked, documented) |

Run the cells in order. Each stage saves its outputs before the next one loads anything.

## 1. Hardware and environment, before any weights load

In [ ]:
# Record the machine first. A run with no GPU assigned is a different experiment.
import os
# Must be set before anything initialises CUDA, so it lives in the very first cell.
# Reduces allocator fragmentation, which matters when a 9.52 GiB model has to fit
# in 14.46 GiB alongside transient activations.
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

import subprocess, sys, platform, json

print("=== nvidia-smi ===")
try:
    print(subprocess.run(["nvidia-smi"], capture_output=True, text=True, timeout=60).stdout)
except FileNotFoundError:
    print("nvidia-smi NOT FOUND -- no GPU is attached to this runtime.")
    print("Runtime > Change runtime type > T4 GPU, then rerun from this cell.")

print("=== system ===")
print("python  :", sys.version.split()[0])
print("platform:", platform.platform())
print(subprocess.run(["free", "-g"], capture_output=True, text=True).stdout or "(no `free`)")
print(subprocess.run(["df", "-h", "/content"], capture_output=True, text=True).stdout)

try:
    import torch
    print("torch       :", torch.__version__)
    print("torch cuda  :", torch.version.cuda)
    print("cuda avail  :", torch.cuda.is_available())
    if torch.cuda.is_available():
        p = torch.cuda.get_device_properties(0)
        free, total = torch.cuda.mem_get_info(0)
        print(f"gpu         : {p.name} (sm_{p.major}{p.minor})")
        print(f"vram        : {free/1024**3:.2f} GiB free / {total/1024**3:.2f} GiB total")
        print("bf16 support:", torch.cuda.is_bf16_supported())
except ImportError:
    print("torch not installed yet")

## 2. Pinned dependencies

`torch` is deliberately **not** pinned: Colab ships a build matched to its CUDA driver, and
replacing it is the most common way to break a runtime.

These pins have been checked for mutual compatibility on PyPI metadata only. **They have not
been executed on a GPU.** If pip resolves something different, cell 3 records what actually
got installed, which is the version set that matters.

In [ ]:
# torch is intentionally absent: keep Colab's own CUDA-matched build.
PINS = [
    "diffusers==0.36.0",
    "transformers==4.57.6",
    "accelerate==1.15.0",
    "huggingface_hub>=0.34,<1.0",
    "sentencepiece==0.2.2",
    "imageio-ffmpeg==0.8.1",
    "imageio==2.37.0",
]

# Install via argv, never through a shell. A pin containing "<" is a redirection
# operator to bash, which silently drops the requirement.
import sys, subprocess
print("Requested pins:")
for pin in PINS:
    print("  ", pin)

proc = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", *PINS],
    capture_output=True, text=True,
)
print(proc.stdout.strip())
print(proc.stderr.strip())
if proc.returncode != 0:
    raise SystemExit(f"pip failed with code {proc.returncode}; read the output above.")
print("\npip finished cleanly. Restart the runtime only if pip explicitly says to.")

In [ ]:
# Record the versions that actually resolved. This is the reproducibility record.
import importlib
RESOLVED = {}
for name in ["torch", "torchvision", "diffusers", "transformers", "accelerate",
             "huggingface_hub", "safetensors", "numpy", "PIL", "imageio",
             "imageio_ffmpeg", "sentencepiece"]:
    try:
        RESOLVED[name] = getattr(importlib.import_module(name), "__version__", "unknown")
    except Exception as exc:
        RESOLVED[name] = f"MISSING ({type(exc).__name__})"
for k, v in RESOLVED.items():
    print(f"{k:18s} {v}")

## 3. Project helpers and the source clip

Two ways to get the helpers. Set `REPO_URL` to your GitHub remote once you have pushed, or
leave it as `None` to upload `src/`, `configs/` and `templates/` by hand.

The clip is **downloaded from Mendeley**, not uploaded. It never needs to pass through the
browser and never enters a git repository.

The clip shows identifiable factory workers who consented to the original study. Keep it,
and anything derived from it, out of public repositories and public posts.

In [ ]:
import os, sys, subprocess, hashlib, json
from pathlib import Path

REPO_URL = None          # e.g. "https://github.com/<you>/unsafenet-counterfactuals.git"
ROOT = Path("/content/unsafenet-counterfactuals")

if REPO_URL and not ROOT.exists():
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(ROOT)], check=True)
elif REPO_URL:
    subprocess.run(["git", "-C", str(ROOT), "pull", "--ff-only"], check=True)

# Fallback: unzip the bundle if it was uploaded to /content.
BUNDLE = Path("/content/unsafenet_cf_bundle.zip")
if not (ROOT / "src" / "unsafenet_cf").exists() and BUNDLE.exists():
    import zipfile
    ROOT.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(BUNDLE) as z:
        z.extractall(ROOT)
    print("unpacked", BUNDLE.name, "into", ROOT)

if not (ROOT / "src" / "unsafenet_cf").exists():
    raise SystemExit(
        f"Helpers not found under {ROOT}.\n"
        "Pick one:\n"
        "  (a) set REPO_URL above to your GitHub remote, or\n"
        "  (b) drag unsafenet_cf_bundle.zip into the Colab file browser at /content, "
        "then rerun this cell."
    )

sys.path.insert(0, str(ROOT / "src"))
os.chdir(ROOT)
from unsafenet_cf import preprocess as pp, manifest as mf, compare as cmp
print("helpers loaded from", ROOT / "src")

OUT = ROOT / "outputs"
(OUT / "preprocessed").mkdir(parents=True, exist_ok=True)
(OUT / "stage3_vae").mkdir(parents=True, exist_ok=True)
(OUT / "stage4_edit").mkdir(parents=True, exist_ok=True)

In [ ]:
# Fetch the single source clip straight from Mendeley and verify its hash.
CLIP_CFG = json.loads((ROOT / "configs" / "clip_0_tr143.json").read_text())
EXP_CFG  = json.loads((ROOT / "configs" / "experiment.json").read_text())

raw = ROOT / "data" / "raw" / "0_tr143.mp4"
raw.parent.mkdir(parents=True, exist_ok=True)

if not raw.exists():
    subprocess.run(["curl", "-sSL", "-o", str(raw), CLIP_CFG["source_download_url"]], check=True)

digest = mf.sha256_file(raw)
if digest != CLIP_CFG["source_sha256"]:
    raise SystemExit(
        f"SHA-256 mismatch for {raw}\n"
        f"  expected {CLIP_CFG['source_sha256']}\n"
        f"  got      {digest}\n"
        "The download is incomplete or the record changed. Do not continue."
    )
print(f"{raw.name}: {raw.stat().st_size} bytes, sha256 verified")
print("unsafe label :", CLIP_CFG["unsafe_label"])
print("safe target  :", CLIP_CFG["target_safe_label"])

## 4. Preprocess to the checkpoint's input shape

Constraints read from `THUDM/CogVideoX-2b` itself, not from memory:

| Source | Value |
|---|---|
| `vae/config.json` | `sample_height=480`, `sample_width=720`, `temporal_compression_ratio=4`, `latent_channels=16`, `scaling_factor=1.15258426` |
| `transformer/config.json` | `sample_frames=49`, `sample_height=60`, `sample_width=90`, `patch_size=2` |

So 49 pixel frames become 13 latent frames, and 720×480 pixels become 90×60 latents.

The source is 1920×1080 at 25 fps. A 1620×1080 crop at `x=300` gives the 3:2 aspect the
checkpoint needs while keeping both the worker and the green walkway. A centred crop
(`x=150`) would cut into the walkway. Every third source frame from 0 to 144 gives 49 frames
spanning 5.88 s of real time, written out at 8 fps.

In [ ]:
spec = pp.load_spec(ROOT / "configs" / "clip_0_tr143.json")
pre_path = OUT / "preprocessed" / "0_tr143_720x480x49.mp4"
record = pp.preprocess_clip(spec, pre_path)

print("source:", {k: record["source_probe"][k] for k in
                  ("width", "height", "fps", "frames", "duration_s")})
print("output:", {k: record["output_probe"][k] for k in
                  ("width", "height", "fps", "frames", "duration_s")})
print("latent shape (C,T,H,W):",
      (pp.LATENT_CHANNELS, record["spec"]["latent_frames"],
       record["spec"]["latent_height"], record["spec"]["latent_width"]))
(OUT / "preprocessed" / "preprocess_record.json").write_text(json.dumps(record, indent=2))

from diffusers.utils import load_video
SOURCE_FRAMES = load_video(str(pre_path))
assert len(SOURCE_FRAMES) == 49, f"expected 49 frames, got {len(SOURCE_FRAMES)}"
print("loaded", len(SOURCE_FRAMES), "frames of size", SOURCE_FRAMES[0].size)
SOURCE_FRAMES[0]

## 5. Stage 3: VAE reconstruction check

Load **only** `AutoencoderKLCogVideoX`, with no transformer and no text encoder. Encode the
preprocessed clip and decode it straight back. No diffusion, no editing.

This answers one question: does the VAE alone preserve the details that decide the label —
the worker's outline, the green walkway edge, and the yellow boundary lines? If detail is
already lost here, no amount of prompt tuning in Stage 4 can recover it.

`.mode()` is used instead of `.sample()` so the reconstruction is deterministic.

In [ ]:
import time, torch, numpy as np

DTYPE = torch.float16   # CogVideoX-2b is documented with fp16; a T4 has no bf16.
DEVICE = "cuda"

mf.reset_peak_memory()
t0 = time.time()

from diffusers import AutoencoderKLCogVideoX
vae = AutoencoderKLCogVideoX.from_pretrained(
    EXP_CFG["checkpoint"], subfolder="vae", torch_dtype=DTYPE
).to(DEVICE)
vae.eval()
vae.enable_tiling()    # 49x480x720 does not fit on a 16 GB card untiled
vae.enable_slicing()
load_s = time.time() - t0
print(f"VAE loaded in {load_s:.1f}s; scaling_factor={vae.config.scaling_factor}")

def frames_to_tensor(frames):
    # (T,H,W,3) uint8 PIL frames -> (1,3,T,H,W) float in [-1, 1]
    arr = np.stack([np.asarray(f) for f in frames]).astype(np.float32) / 127.5 - 1.0
    return torch.from_numpy(arr).permute(3, 0, 1, 2).unsqueeze(0)

def tensor_to_frames(x):
    # (1,3,T,H,W) in [-1, 1] -> list of uint8 (H,W,3) arrays
    x = ((x.float().clamp(-1, 1) + 1.0) * 127.5).round().to(torch.uint8)
    return list(x[0].permute(1, 2, 3, 0).cpu().numpy())

t0 = time.time()
with torch.no_grad():
    x = frames_to_tensor(SOURCE_FRAMES).to(DEVICE, dtype=DTYPE)
    latent = vae.encode(x).latent_dist.mode()
    scaled = latent * vae.config.scaling_factor          # invert_scale_latents=false
    recon = vae.decode(scaled / vae.config.scaling_factor).sample
roundtrip_s = time.time() - t0
peak = mf.peak_memory_gb()

print("input tensor  :", tuple(x.shape))
print("latent tensor :", tuple(latent.shape), "<- expect (1, 16, 13, 60, 90)")
print(f"round trip    : {roundtrip_s:.1f}s, peak GPU {peak} GiB")

RECON_FRAMES = tensor_to_frames(recon)
src_arr = cmp.to_array(SOURCE_FRAMES)
rec_arr = np.stack(RECON_FRAMES)
print(f"PSNR source vs reconstruction: {cmp.psnr(src_arr, rec_arr):.2f} dB")
print(f"mean |diff|                  : {cmp.mean_abs_diff(src_arr, rec_arr):.2f} / 255")

In [ ]:
# Save the reconstruction and a side-by-side sheet, then release the VAE.
from diffusers.utils import export_to_video
from PIL import Image

recon_path = OUT / "stage3_vae" / "reconstruction.mp4"
export_to_video([Image.fromarray(f) for f in RECON_FRAMES], str(recon_path), fps=8)

grid = cmp.comparison_grid(
    [("source", src_arr), ("vae_reconstruction", rec_arr)],
    out_path=OUT / "stage3_vae" / "source_vs_reconstruction.png",
)
print("rows:", grid["order"], "frames:", grid["frame_indices"])

per_frame = cmp.per_frame_psnr(src_arr, rec_arr)
mf.write_manifest(
    mf.build_run_manifest(
        stage="vae_reconstruction",
        environment={**mf.environment_snapshot(), "resolved_packages": RESOLVED},
        settings={"checkpoint": EXP_CFG["checkpoint"], "subfolder": "vae",
                  "dtype": str(DTYPE), "tiling": True, "slicing": True,
                  "latent_mode": "latent_dist.mode()",
                  "scaling_factor": vae.config.scaling_factor},
        inputs={"preprocessed_clip": str(pre_path), "preprocess_record": record["spec"]},
        outputs={"reconstruction": str(recon_path),
                 "comparison_sheet": str(grid["path"]),
                 "psnr_db": cmp.psnr(src_arr, rec_arr),
                 "mean_abs_diff": cmp.mean_abs_diff(src_arr, rec_arr),
                 "per_frame_psnr_db": per_frame},
        timing={"vae_load_s": load_s, "roundtrip_s": roundtrip_s},
        memory={"peak_gpu_gib": peak},
        notes="VAE only. No diffusion, no editing, no safety claim.",
    ),
    OUT / "stage3_vae" / "manifest.json",
)

del vae, x, latent, scaled, recon
mf.free_gpu()
print("VAE released; GPU peak counter reset for Stage 4.")
grid["image"]

**Inspect the sheet above before continuing.** Top row is the preprocessed source, bottom row
is the VAE round trip. Look specifically at:

- the worker's silhouette and feet, which decide where he is standing;
- the green walkway edge and the yellow boundary lines;
- text and signage on the machines, which is the first thing a VAE smears.

Record what you see in `templates/REVIEW_TEMPLATE.md`. A high PSNR with a smeared walkway edge
is still a failure for this experiment.

## 6. Stage 4: prompt-guided source-video edit

`CogVideoXVideoToVideoPipeline` encodes the **actual source video** into latents, adds noise
scaled by `strength`, and denoises with text conditioning. It does not start from random noise
and it does not start from a single frame.

Signature confirmed against the diffusers docs:
`__call__(video, prompt, negative_prompt, height, width, num_inference_steps=50, strength=0.8, guidance_scale=6, ...)`.
There is **no `num_frames` argument** — the frame count comes from `len(video)`, which is 49.

Memory note, measured on this runtime: the GPU has 14.46 GiB but the **system has only
12 GiB of RAM**, and the checkpoint totals 13.78 GiB (text_encoder 9.52, transformer 3.39,
vae 0.86). `enable_model_cpu_offload()` holds every weight in system RAM, so loading the
whole pipeline at once exceeds RAM and kills the session.

The fix is two-phase loading. Phase A puts the T5-XXL encoder on the **GPU** alone, where
9.52 GiB fits in 14.46 GiB, encodes the two prompt strings, and releases it. Phase B loads
the pipeline with `text_encoder=None` and reuses the embeddings, so only 4.25 GiB of weights
ever sit in RAM.

If a stage still runs out of GPU memory, the documented fallback is
`enable_sequential_cpu_offload()`, which is far slower. Change one thing at a time and
record what you changed.


In [ ]:
# Phase A: load ONLY the text encoder, encode the prompts, then release it.
#
# Why two phases: this runtime has 12 GiB of system RAM and the checkpoint totals 13.78 GiB
# (text_encoder 9.52, transformer 3.39, vae 0.86). enable_model_cpu_offload() keeps every
# weight in system RAM, so a single-phase load would exceed RAM and kill the session.
#
# Why no_grad: encode_prompt() is a plain method. Only the pipeline's __call__ wraps itself
# in torch.no_grad(), so calling encode_prompt() directly leaves autograd on and T5-XXL
# retains a backward graph it never needs. That is what exhausted a 14.56 GiB card.
import gc, sys, time, torch
from diffusers import CogVideoXVideoToVideoPipeline

# Release anything a previous failed attempt is still holding, including the
# exception traceback, which keeps every local frame of the failed call alive.
for _name in ("text_pipe", "text_encoder"):
    globals().pop(_name, None)
for _attr in ("last_traceback", "last_value", "last_type"):
    if hasattr(sys, _attr):
        try:
            delattr(sys, _attr)
        except AttributeError:
            pass
gc.collect()
mf.free_gpu()

free0, total = torch.cuda.mem_get_info()
print(f"free VRAM before load : {free0/1024**3:.2f} GiB of {total/1024**3:.2f} GiB")

# The text encoder needs 9.52 GiB of weights plus activation headroom. A failed
# earlier attempt can leave allocations pinned by references this cell cannot reach
# (IPython output history, traceback frames held by the kernel). Fail fast and say
# so, rather than OOM again inside .to() and look like the same bug twice.
REQUIRED_FREE_GIB = 11.0
if free0 / 1024 ** 3 < REQUIRED_FREE_GIB:
    raise SystemExit(
        f"Only {free0/1024**3:.2f} GiB of VRAM is free; this stage needs "
        f"{REQUIRED_FREE_GIB} GiB.\n"
        "A previous attempt is still holding GPU memory that cannot be released "
        "from inside the kernel.\n"
        "Fix: Runtime > Restart session, then rerun every cell from the top. "
        "Pip packages and downloaded weights survive a restart, so this costs "
        "about a minute, not another download."
    )

t0 = time.time()
text_pipe = CogVideoXVideoToVideoPipeline.from_pretrained(
    EXP_CFG["checkpoint"], transformer=None, vae=None, torch_dtype=DTYPE
)
param_dtype = next(text_pipe.text_encoder.parameters()).dtype
print("text encoder param dtype:", param_dtype)
assert param_dtype == DTYPE, f"expected {DTYPE}, got {param_dtype}"

text_pipe.text_encoder.to(DEVICE)
free1, _ = torch.cuda.mem_get_info()
print(f"weights resident      : {(free0 - free1)/1024**3:.2f} GiB")
print(f"free VRAM after load  : {free1/1024**3:.2f} GiB")

mf.reset_peak_memory()
with torch.no_grad():
    prompt_embeds, negative_prompt_embeds = text_pipe.encode_prompt(
        prompt=EXP_CFG["prompt"],
        negative_prompt=EXP_CFG["negative_prompt"],
        do_classifier_free_guidance=True,
        num_videos_per_prompt=1,
        max_sequence_length=226,
        device=DEVICE,
        dtype=DTYPE,
    )
encode_s = time.time() - t0
peak_encode = mf.peak_memory_gb()

# Keep the embeddings (about 1.9 MB each), drop the 9.52 GiB encoder.
prompt_embeds = prompt_embeds.detach().cpu()
negative_prompt_embeds = negative_prompt_embeds.detach().cpu()

print("prompt_embeds         :", tuple(prompt_embeds.shape), "<- expect (1, 226, 4096)")
print("negative_prompt_embeds:", tuple(negative_prompt_embeds.shape))
print(f"encoded in {encode_s:.1f}s, peak GPU {peak_encode} GiB")

del text_pipe
mf.free_gpu()
free2, _ = torch.cuda.mem_get_info()
print(f"text encoder released; free VRAM now {free2/1024**3:.2f} GiB")

In [ ]:
# Phase B: load the pipeline WITHOUT the text encoder and reuse the embeddings.
import time, torch, subprocess
from diffusers import CogVideoXVideoToVideoPipeline

mf.reset_peak_memory()
t0 = time.time()
pipe = CogVideoXVideoToVideoPipeline.from_pretrained(
    EXP_CFG["checkpoint"], text_encoder=None, tokenizer=None, torch_dtype=DTYPE
)
pipe.enable_model_cpu_offload()    # do NOT also call .to("cuda")
pipe.vae.enable_tiling()
pipe.vae.enable_slicing()
pipe_load_s = time.time() - t0

print(f"pipeline loaded in {pipe_load_s:.1f}s (text encoder omitted)")
print("scheduler :", type(pipe.scheduler).__name__)
print("prediction:", pipe.scheduler.config.prediction_type)
print(subprocess.run(["free", "-g"], capture_output=True, text=True).stdout)

In [ ]:
SEED = EXP_CFG["seed"]
generator = torch.Generator(device="cpu").manual_seed(SEED)

print("prompt:", EXP_CFG["prompt"])
t0 = time.time()
result = pipe(
    video=SOURCE_FRAMES,
    prompt_embeds=prompt_embeds.to(DEVICE),
    negative_prompt_embeds=negative_prompt_embeds.to(DEVICE),
    height=pp.TARGET_HEIGHT,
    width=pp.TARGET_WIDTH,
    num_inference_steps=EXP_CFG["num_inference_steps"],
    strength=EXP_CFG["strength"],
    guidance_scale=EXP_CFG["guidance_scale"],
    generator=generator,
)
gen_s = time.time() - t0
peak_edit = mf.peak_memory_gb()

EDIT_FRAMES = result.frames[0]
print(f"generated {len(EDIT_FRAMES)} frames in {gen_s:.1f}s, peak GPU {peak_edit} GiB")

edit_path = OUT / "stage4_edit" / f"edit_seed{SEED}_s{EXP_CFG['strength']}.mp4"
export_to_video(EDIT_FRAMES, str(edit_path), fps=8)
print("saved", edit_path)

In [ ]:
# Comparison sheet across all three clips, plus the stage manifest.
import numpy as np

edit_arr = cmp.to_array(EDIT_FRAMES)
grid4 = cmp.comparison_grid(
    [("source", src_arr), ("vae_reconstruction", rec_arr), ("edited", edit_arr)],
    out_path=OUT / "stage4_edit" / "source_vs_recon_vs_edit.png",
)
print("rows:", grid4["order"], "frames:", grid4["frame_indices"])

mf.write_manifest(
    mf.build_run_manifest(
        stage="prompt_guided_edit",
        environment={**mf.environment_snapshot(), "resolved_packages": RESOLVED},
        settings={
            "checkpoint": EXP_CFG["checkpoint"],
            "revision": EXP_CFG["revision"],
            "dtype": str(DTYPE),
            "scheduler": type(pipe.scheduler).__name__,
            "prediction_type": pipe.scheduler.config.prediction_type,
            "seed": SEED,
            "generator_device": "cpu",
            "num_inference_steps": EXP_CFG["num_inference_steps"],
            "strength": EXP_CFG["strength"],
            "guidance_scale": EXP_CFG["guidance_scale"],
            "prompt": EXP_CFG["prompt"],
            "negative_prompt": EXP_CFG["negative_prompt"],
            "num_frames": len(SOURCE_FRAMES),
            "height": pp.TARGET_HEIGHT,
            "width": pp.TARGET_WIDTH,
            "optimizations": ["two-phase text encoding (RAM limit)",
                              "enable_model_cpu_offload", "vae tiling", "vae slicing"],
            "max_sequence_length": 226,
        },
        inputs={
            "preprocessed_clip": str(pre_path),
            "source_sha256": CLIP_CFG["source_sha256"],
            "unsafe_label": CLIP_CFG["unsafe_label"],
            "target_safe_label": CLIP_CFG["target_safe_label"],
            "context_that_must_not_change": EXP_CFG["context_that_must_not_change"],
            "spurious_change_to_reject": EXP_CFG["spurious_change_to_reject"],
        },
        outputs={
            "edited_clip": str(edit_path),
            "comparison_sheet": str(grid4["path"]),
            "psnr_source_vs_edit_db": cmp.psnr(src_arr, edit_arr),
            "mean_abs_diff_source_vs_edit": cmp.mean_abs_diff(src_arr, edit_arr),
        },
        timing={"prompt_encode_s": encode_s, "pipeline_load_s": pipe_load_s,
                "generation_s": gen_s},
        memory={"peak_gpu_gib_encode": peak_encode, "peak_gpu_gib": peak_edit},
        notes=("Prompt-guided editing baseline using a pretrained CogVideoX-2b. "
               "Not LD-ViCE: no per-dataset fine-tuning, no classifier gradients, "
               "no SmoothGrad, no artifact refinement. Output is a candidate "
               "counterfactual pending human review."),
    ),
    OUT / "stage4_edit" / "manifest.json",
)
print(f"PSNR source vs edit: {cmp.psnr(src_arr, edit_arr):.2f} dB")
grid4["image"]

In [ ]:
# Save everything off the ephemeral runtime. Do this after every stage.
import shutil
archive = shutil.make_archive("/content/unsafenet_cf_outputs", "zip", str(OUT))
print("archive:", archive, f"{Path(archive).stat().st_size/1e6:.1f} MB")

try:
    from google.colab import files
    files.download(archive)
except Exception as exc:
    print("Automatic download unavailable:", exc)
    print("Download it from the Colab file browser instead.")

# Optional: copy to Drive. Mounting is not required for anything above.
MOUNT_DRIVE = False
if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    dest = Path("/content/drive/MyDrive/unsafenet_cf")
    dest.mkdir(parents=True, exist_ok=True)
    shutil.copy(archive, dest)
    print("copied to", dest)

## 7. Stage 5: classifier-guided feasibility gate — **blocked**

CLAUDE.md gates this stage on a suitable differentiable classifier existing. One does not.

**The blocker, precisely:** the Unsafe-Net paper (doi:10.1007/s11042-024-19276-8) publishes no
code repository and no trained weights for its YOLO v4 + ConvLSTM system. Its data-availability
statement covers the dataset only. The Mendeley record (doi:10.17632/xjmtb22pff.1) contains 691
`.mp4` files and no model artefacts. So there is no classifier whose label space contains
*Safe Walkway* and *Safe Walkway Violation* as the paper defines them.

**What would be required to unblock it**, in order:

1. Train an 8-class classifier on the Mendeley dataset reproducing the paper's label space.
   That is dataset-scale training, outside this feasibility milestone.
2. Verify its label space actually contains class 4 *Safe Walkway*. The absence of class 0
   is **not** equivalent to the presence of class 4.
3. Verify gradients reach the optimised latent through classifier preprocessing, the
   classifier itself, and VAE decoding. Frozen weights must not block input gradients.
4. Replace the stock pipeline loop. `CogVideoXVideoToVideoPipeline.__call__` runs the
   denoising loop under `torch.no_grad()`, so calling it cannot produce classifier guidance
   no matter what is passed in.
5. Keep inference-only optimisations away from the autograd path. `enable_model_cpu_offload`
   and VAE tiling are for the no-grad stages; verify each separately before trusting it
   inside a guided loop.

**Substituting an unrelated classifier would invalidate the experiment** and is not done here.

The cell below is a gradient-path probe for when a classifier exists. It is guarded off and
has never been executed. Running it without a real classifier proves nothing.

In [ ]:
RUN_GRADIENT_PROBE = False   # flip only once a real UnsafeNet classifier exists

if not RUN_GRADIENT_PROBE:
    print("Stage 5 skipped: no target classifier available. See the cell above.")
else:
    # One guided denoising step only. Check finite, nonzero gradients and measure
    # memory and time BEFORE attempting a full guided loop.
    import torch

    classifier = None          # <- load the real classifier here
    target_class = CLIP_CFG["target_safe_label"]["class_id"]
    if classifier is None:
        raise SystemExit("RUN_GRADIENT_PROBE is on but no classifier was loaded.")

    vae = pipe.vae
    vae.enable_tiling()

    latents = ...              # <- one noisy latent from the source encoding
    latents = latents.detach().clone().requires_grad_(True)

    with torch.enable_grad():                     # the stock pipeline disables this
        decoded = vae.decode(latents / vae.config.scaling_factor).sample
        logits = classifier(decoded)
        loss = torch.nn.functional.cross_entropy(
            logits, torch.tensor([target_class], device=logits.device)
        )
        grad, = torch.autograd.grad(loss, latents)

    print("grad shape   :", tuple(grad.shape))
    print("all finite   :", bool(torch.isfinite(grad).all()))
    print("nonzero frac :", float((grad != 0).float().mean()))
    print("grad norm    :", float(grad.norm()))
    print("peak GPU GiB :", mf.peak_memory_gb())
    # Only if all four lines look sane should a full guided loop be attempted.

## 8. What to record before claiming anything

Fill in `templates/REVIEW_TEMPLATE.md`. The four criteria are judged separately:

1. **Did the safety-critical behaviour change?** Is the worker now on the green walkway?
2. **Was context preserved?** Worker identity, task, objects, camera, scene.
3. **Temporal consistency, physical plausibility, artifacts.**
4. **Classifier prediction** — not applicable here; no classifier exists.

Reject as a **spurious counterfactual** if the model repainted, widened or extended the green
walkway underneath the worker rather than moving the worker onto it. That changes the
environment, not the behaviour.

A failed run with a documented, actionable cause is an honest result. It is not a successful
generation, and it must not be written up as one.